In [1]:
from data_prep import DataPrep
import pandas as pd
import yfinance as yf
from portfolio_eval import PriceData, PortfolioSelection, DiversificationAnalysis
import numpy as np

In [2]:
data = DataPrep.full_data_prep(0.7, 0.15, 0.15, '2023-12-31', 1)
test_entities = data['test_entities']

Train Rows: 11219 (5548 Firmen)
Val Rows:   1181 (1189 Firmen)
Test Rows:  1186 (1190 Firmen)


In [3]:
data = []

for symbol in test_entities:
    t = yf.Ticker(symbol)
    info = t.info

    data.append(
        {
            "Ticker": symbol,
            "Name": info.get("shortName", "N/A"),
            "Sektor": info.get("sector", "N/A"),
            "Branche": info.get("industry", "N/A"),
            "Land": info.get("country", "N/A"),
        }
    )
data

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 0431.HK"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 0724.HK"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ALVG.PA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CMU"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DIAD.MI"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EVTV"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MLAST.PA"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote 

[{'Ticker': '0006.HK',
  'Name': 'POWER ASSETS',
  'Sektor': 'Utilities',
  'Branche': 'Utilities - Independent Power Producers',
  'Land': 'Hong Kong'},
 {'Ticker': '0008.HK',
  'Name': 'PCCW',
  'Sektor': 'Communication Services',
  'Branche': 'Telecom Services',
  'Land': 'Hong Kong'},
 {'Ticker': '0026.HK',
  'Name': 'CHINA MOTOR BUS',
  'Sektor': 'Real Estate',
  'Branche': 'Real Estate - Diversified',
  'Land': 'Hong Kong'},
 {'Ticker': '0027.HK',
  'Name': 'GALAXY ENT',
  'Sektor': 'Consumer Cyclical',
  'Branche': 'Resorts & Casinos',
  'Land': 'Hong Kong'},
 {'Ticker': '0033.HK',
  'Name': 'INTL GENIUS',
  'Sektor': 'Financial Services',
  'Branche': 'Asset Management',
  'Land': 'China'},
 {'Ticker': '0038.HK',
  'Name': 'FIRST TRACTOR',
  'Sektor': 'Industrials',
  'Branche': 'Farm & Heavy Construction Machinery',
  'Land': 'China'},
 {'Ticker': '0039.HK',
  'Name': 'CH BEIDAHUANG',
  'Sektor': 'Consumer Defensive',
  'Branche': 'Grocery Stores',
  'Land': 'Hong Kong'},
 {'T

In [4]:
df = pd.DataFrame(data)
df_clean = df[
    df["Branche"].notna() & (~df["Branche"].isin(["N/A", "None", ""]))
].copy()


In [5]:
returns_test = PriceData.execute(df_clean['Ticker'].to_list(), '2024-01-01', '2026-09-30', 0.00)

$MLCAN.PA: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$MLASO.PA: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$0501.HK: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$TALK: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$MVE.OL: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$MLARE.PA: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$HLX: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$MLMAC.LS: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$ELIMP.OL: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$NFBK: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)
$EA: possibly delisted; no price data found  (1d 2024-01-01 -> 2026-09-30)

11 Failed downloads:
['MLCAN.PA', 'MLASO.PA', '0501.HK', 'TALK', 'MVE.OL', 'MLARE.PA', 'HLX', 'MLMAC.LS', 'ELIMP.OL', 'NFBK', 'EA'

Eingangs-Ticker:          1171
Nutzbare Ticker nach 0.0 Filterung: 1063


In [31]:
returns_test.columns

Index(['0006.HK', '0008.HK', '0026.HK', '0027.HK', '0033.HK', '0038.HK',
       '0039.HK', '0053.HK', '0069.HK', '0070.HK',
       ...
       'XMAX', 'XOS', 'XXII', 'YELP', 'YUMC', 'ZAP.OL', 'ZD', 'ZG', 'ZNTL',
       'ZVIA'],
      dtype='object', name='Ticker', length=1064)

In [20]:
df_clean = df_clean[df_clean['Ticker'].isin(returns_test.columns)]

In [24]:


# 2. Funktion zum Ziehen eines Portfolios mit eindeutigen Branchen
def generate_diversified_portfolio(data: pd.DataFrame, n_assets: int):
    # Prüfen, ob überhaupt genügend Branchen vorhanden sind
    data = data[data['Ticker'].isin(returns_test.columns)]
    available_industries = data["Branche"].nunique()
    if n_assets > available_industries:
        raise ValueError(
            f"Anforderung nicht möglich: Du möchtest {n_assets} Titel, aber es gibt nur "
            f"{available_industries} verschiedene Branchen im Datensatz."
        )

    # Aus jeder Branche zufällig 1 Aktie ziehen
    one_per_industry = data.groupby("Branche", group_keys=False).apply(
        lambda x: x.sample(1, random_state=None)
    )

    # Aus den gezogenen Branchen-Vertretern zufällig 'n_assets' auswählen
    portfolio = one_per_industry.sample(n=n_assets).reset_index(drop=True)

    return portfolio['Ticker'].to_list()
erg = {}
volatility = {}
mean_vol = {}
for N in [10,30,50]:
    erge = []
    volas = []
    mean_vola = []
    for i in range(10000):
        portfolio = generate_diversified_portfolio(df_clean, N)
        ergeb = DiversificationAnalysis.diversification_ratio(returns_test, portfolio)
        erge.append(ergeb['diversification_ratio'])
        volas.append(ergeb['portfolio_volatility_annual'])
        mean_vola.append(ergeb['weighted_avg_volatility_annual'])
    erg[N] = (np.mean(erge), np.std(erge))
    volatility[N] = (np.mean(volas), np.std(volas))
    mean_vol[N] = (np.mean(mean_vola), np.std(mean_vola))
        

In [26]:
for N in [10,30,50]:
    print(f'N={N}: DivRatio={erg[N][0]} | STD={erg[N][1]}')
    print(f'N={N}: Volatility={volatility[N][0]} | STD={volatility[N][1]}')
    print(f'N={N}: Single AVG Volatility={mean_vol[N][0]} | STD={mean_vol[N][1]}')
    print("-----------------------------------------------------------")

N=10: DivRatio=2.3363831930862604 | STD=0.2629945327548761
N=10: Volatility=0.314443444814356 | STD=0.6422448938184648
N=10: Single AVG Volatility=0.6353886857178916 | STD=0.672630442853178
-----------------------------------------------------------
N=30: DivRatio=3.2531458230726455 | STD=0.42834352986422075
N=30: Volatility=0.2254613608850937 | STD=0.3134398672295945
N=30: Single AVG Volatility=0.6218349836383894 | STD=0.34435579632831503
-----------------------------------------------------------
N=50: DivRatio=3.6223892161427744 | STD=0.587007620784803
N=50: Volatility=0.2091008180394584 | STD=0.2449634585622663
N=50: Single AVG Volatility=0.6263708701640683 | STD=0.28004814298026603
-----------------------------------------------------------


In [18]:
erg[10]

(2.3434643772007555, 0.25620791126222303)

In [22]:
df_clean['Branche'].nunique()

133

In [23]:
df_clean['Land'].nunique()

33